In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

In [3]:
INTERIM_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# CSV 1: dailyActivity_merged.csv

In [4]:
daily_activity = pd.read_csv(
    RAW_DATA_PATH / "dailyActivity_merged.csv"
)

In [6]:
daily_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457 entries, 0 to 456
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Id                        457 non-null    int64  
 1   ActivityDate              457 non-null    object 
 2   TotalSteps                457 non-null    int64  
 3   TotalDistance             457 non-null    float64
 4   TrackerDistance           457 non-null    float64
 5   LoggedActivitiesDistance  457 non-null    float64
 6   VeryActiveDistance        457 non-null    float64
 7   ModeratelyActiveDistance  457 non-null    float64
 8   LightActiveDistance       457 non-null    float64
 9   SedentaryActiveDistance   457 non-null    float64
 10  VeryActiveMinutes         457 non-null    int64  
 11  FairlyActiveMinutes       457 non-null    int64  
 12  LightlyActiveMinutes      457 non-null    int64  
 13  SedentaryMinutes          457 non-null    int64  
 14  Calories  

In [7]:
daily_activity.columns = (
    daily_activity.columns
    .str.strip()
    .str.replace(r'([a-z0-9])([A-Z])', r'\1_\2', regex=True)
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

In [8]:
daily_activity.columns.tolist()

['id',
 'activity_date',
 'total_steps',
 'total_distance',
 'tracker_distance',
 'logged_activities_distance',
 'very_active_distance',
 'moderately_active_distance',
 'light_active_distance',
 'sedentary_active_distance',
 'very_active_minutes',
 'fairly_active_minutes',
 'lightly_active_minutes',
 'sedentary_minutes',
 'calories']

In [9]:
daily_activity['id'] = daily_activity['id'].astype('string')

In [10]:
daily_activity['activity_date'] = pd.to_datetime(
    daily_activity['activity_date'],
    format='%m/%d/%Y',
    errors='coerce'
)

In [11]:
print(daily_activity.dtypes)
print("Invalid dates:", daily_activity['activity_date'].isna().sum())

id                            string[python]
activity_date                 datetime64[ns]
total_steps                            int64
total_distance                       float64
tracker_distance                     float64
logged_activities_distance           float64
very_active_distance                 float64
moderately_active_distance           float64
light_active_distance                float64
sedentary_active_distance            float64
very_active_minutes                    int64
fairly_active_minutes                  int64
lightly_active_minutes                 int64
sedentary_minutes                      int64
calories                               int64
dtype: object
Invalid dates: 0


In [12]:
print("Dataset shape:", daily_activity.shape)

print(
    "Date range:",
    daily_activity['activity_date'].min(),
    "to",
    daily_activity['activity_date'].max()
)

print("Unique participants:", daily_activity['id'].nunique())

print("Missing values:", daily_activity.isna().sum().sum())

print("Exact duplicate rows:", daily_activity.duplicated().sum())

print(
    "Duplicate participant-date combinations:",
    daily_activity.duplicated(
        subset=['id', 'activity_date']
    ).sum()
)

Dataset shape: (457, 15)
Date range: 2016-03-12 00:00:00 to 2016-04-12 00:00:00
Unique participants: 35
Missing values: 0
Exact duplicate rows: 0
Duplicate participant-date combinations: 0


In [16]:
daily_activity_clean = daily_activity.copy()

In [20]:
## Create a daily completeness indicator

In [21]:
minute_cols = [
    'very_active_minutes',
    'fairly_active_minutes',
    'lightly_active_minutes',
    'sedentary_minutes'
]

daily_activity_clean['total_recorded_minutes'] = (
    daily_activity_clean[minute_cols].sum(axis=1)
)

daily_activity_clean['is_complete_day'] = (
    daily_activity_clean['total_recorded_minutes'] == 1440
)

In [22]:
## Create a distance discrepancy indicator

In [23]:
distance_cols = [
    'very_active_distance',
    'moderately_active_distance',
    'light_active_distance',
    'sedentary_active_distance'
]

daily_activity_clean['component_distance_sum'] = (
    daily_activity_clean[distance_cols].sum(axis=1)
)

daily_activity_clean['distance_discrepancy'] = (
    ~np.isclose(
        daily_activity_clean['total_distance'],
        daily_activity_clean['component_distance_sum'],
        atol=0.01
    )
)

In [24]:
## Validate the transformations

In [25]:
print("Total records:", len(daily_activity_clean))

print(
    "Complete days:",
    daily_activity_clean['is_complete_day'].sum()
)

print(
    "Incomplete days:",
    (~daily_activity_clean['is_complete_day']).sum()
)

print(
    "Distance discrepancy records:",
    daily_activity_clean['distance_discrepancy'].sum()
)

Total records: 457
Complete days: 233
Incomplete days: 224
Distance discrepancy records: 65


In [26]:
daily_activity_clean.to_csv(
    PROCESSED_DATA_PATH / "daily_activity_clean.csv",
    index=False
)

In [27]:
daily_activity_clean

,id,activity_date,total_steps,total_distance,tracker_distance,logged_activities_distance,very_active_distance,moderately_active_distance,light_active_distance,sedentary_active_distance,very_active_minutes,fairly_active_minutes,lightly_active_minutes,sedentary_minutes,calories,total_recorded_minutes,is_complete_day,component_distance_sum,distance_discrepancy
0,1503960366,2016-03-25,11004,7.110000,7.110000,0.0,2.57,0.46,4.07,0.00,33,12,205,804,1819,1054,False,7.10,False
1,1503960366,2016-03-26,17609,11.550000,11.550000,0.0,6.92,0.73,3.91,0.00,89,17,274,588,2154,968,False,11.56,False
2,1503960366,2016-03-27,12736,8.530000,8.530000,0.0,4.66,0.16,3.71,0.00,56,5,268,605,1944,934,False,8.53,False
3,1503960366,2016-03-28,13231,8.930000,8.930000,0.0,3.19,0.79,4.95,0.00,39,20,224,1080,1932,1363,False,8.93,False
4,1503960366,2016-03-29,12041,7.850000,7.850000,0.0,2.16,1.09,4.61,0.00,28,28,243,763,1886,1062,False,7.86,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452,8877689391,2016-04-08,23014,20.389999,20.389999,0.0,11.10,0.63,8.62,0.00,70,29,359,982,4196,1440,True,20.35,True
453,8877689391,2016-04-09,16470,8.070000,8.070000,0.0,0.00,0.02,8.02,0.00,90,9,289,1052,3841,1440,True,8.04,True
454,8877689391,2016-04-10,28497,27.530001,27.530001,0.0,21.92,1.12,4.46,0.00,128,46,211,1055,4526,1440,True,27.50,True
455,8877689391,2016-04-11,10622,8.060000,8.060000,0.0,1.47,0.15,6.37,0.01,18,7,225,1190,2820,1440,True,8.00,True


# CSV 2 — minuteSleep_merged.csv

In [28]:
sleep = pd.read_csv(
    RAW_DATA_PATH / "minuteSleep_merged.csv"
)

In [29]:
sleep.columns = (
    sleep.columns
    .str.strip()
    .str.replace(
        r'([a-z0-9])([A-Z])',
        r'\1_\2',
        regex=True
    )
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

In [30]:
sleep['id'] = sleep['id'].astype('string')

In [31]:
sleep['date'] = pd.to_datetime(
    sleep['date'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)

In [32]:
print("Invalid timestamps:", sleep['date'].isna().sum())
print("Data type:", sleep['date'].dtype)

Invalid timestamps: 0
Data type: datetime64[ns]


In [33]:
## Remove confirmed duplicate records

In [34]:
sleep_clean = sleep.drop_duplicates(
    subset=['id', 'log_id', 'date'],
    keep='first'
).copy()

In [35]:
print("Original records:", len(sleep))
print("Cleaned records:", len(sleep_clean))
print("Records removed:", len(sleep) - len(sleep_clean))

print(
    "Remaining duplicate keys:",
    sleep_clean.duplicated(
        subset=['id', 'log_id', 'date']
    ).sum()
)

Original records: 198559
Cleaned records: 198034
Records removed: 525
Remaining duplicate keys: 0


In [38]:
## sleep calendar-date coverage per participan

In [39]:
sleep_clean['calendar_date'] = (
    sleep_clean['date'].dt.normalize()
)

In [40]:
sleep_coverage = (
    sleep_clean.groupby('id')
    .agg(
        total_records=('date', 'size'),
        unique_sleep_logs=('log_id', 'nunique'),
        unique_calendar_dates=('calendar_date', 'nunique'),
        first_record=('date', 'min'),
        last_record=('date', 'max')
    )
    .reset_index()
)

In [44]:
sleep_coverage.sort_values(
    'unique_calendar_dates',
    ascending=False
)

,id,total_records,unique_sleep_logs,unique_calendar_dates,first_record,last_record
5,2026352035,17504,32,33,2016-03-11 21:31:30,2016-04-12 05:53:00
17,6962181067,14903,36,33,2016-03-11 23:29:00,2016-04-12 06:49:30
21,8378563200,15111,49,33,2016-03-11 22:08:30,2016-04-12 04:14:00
14,5577150313,13885,44,32,2016-03-11 21:19:30,2016-04-11 05:32:00
3,1927972279,11513,41,31,2016-03-11 23:24:00,2016-04-12 08:35:00
6,2347167796,13484,28,31,2016-03-11 23:43:00,2016-04-11 06:38:00
12,4702921684,13148,32,30,2016-03-11 21:56:30,2016-04-11 05:55:30
13,5553957443,14602,40,30,2016-03-12 00:36:30,2016-04-12 06:03:00
9,4319703577,12426,27,28,2016-03-11 22:30:30,2016-04-07 07:29:30
15,6117666160,12925,44,27,2016-03-11 22:42:30,2016-04-09 23:04:00


In [45]:
sleep_clean.to_csv(
    PROCESSED_DATA_PATH / 'minute_sleep_clean.csv',
    index=False
)

# CSV 3 — weightLogInfo_merged.csv

In [46]:
weight = pd.read_csv(
    RAW_DATA_PATH / 'weightLogInfo_merged.csv'
)

weight.columns = (
    weight.columns
    .str.strip()
    .str.replace(
        r'([a-z0-9])([A-Z])',
        r'\1_\2',
        regex=True
    )
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

In [47]:
weight['id'] = weight['id'].astype('string')

weight['date'] = pd.to_datetime(
    weight['date'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)

weight['log_id'] = weight['log_id'].astype('string')

In [48]:
weight_clean = weight.drop(
    columns=['fat']
).copy()

In [49]:
print("Shape:", weight_clean.shape)

print("Columns:", weight_clean.columns.tolist())

print("Missing values:")
print(weight_clean.isna().sum())

Shape: (33, 7)
Columns: ['id', 'date', 'weight_kg', 'weight_pounds', 'bmi', 'is_manual_report', 'log_id']
Missing values:
id                  0
date                0
weight_kg           0
weight_pounds       0
bmi                 0
is_manual_report    0
log_id              0
dtype: int64


In [50]:
weight_clean.to_csv(
    PROCESSED_DATA_PATH / 'weight_clean.csv',
    index=False
)

# CSV 4 — hourlyCalories_merged.csv

In [51]:
hourly_calories = pd.read_csv(
    RAW_DATA_PATH / 'hourlyCalories_merged.csv'
)

In [53]:
hourly_calories.columns = (
    hourly_calories.columns
    .str.strip()
    .str.replace(
        r'([a-z0-9])([A-Z])',
        r'\1_\2',
        regex=True
    )
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

In [54]:
hourly_calories['id'] = (
    hourly_calories['id'].astype('string')
)

hourly_calories['activity_hour'] = pd.to_datetime(
    hourly_calories['activity_hour'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)

In [55]:
hourly_calories['date'] = (
    hourly_calories['activity_hour'].dt.normalize()
)

In [56]:
hourly_calories

,id,activity_hour,calories,date
0,1503960366,2016-03-12 00:00:00,48,2016-03-12
1,1503960366,2016-03-12 01:00:00,48,2016-03-12
2,1503960366,2016-03-12 02:00:00,48,2016-03-12
3,1503960366,2016-03-12 03:00:00,48,2016-03-12
4,1503960366,2016-03-12 04:00:00,48,2016-03-12
...,...,...,...,...
24079,8877689391,2016-04-12 04:00:00,73,2016-04-12
24080,8877689391,2016-04-12 05:00:00,73,2016-04-12
24081,8877689391,2016-04-12 06:00:00,96,2016-04-12
24082,8877689391,2016-04-12 07:00:00,169,2016-04-12


In [57]:
hourly_intensities = pd.read_csv(
    RAW_DATA_PATH / 'hourlyIntensities_merged.csv'
)

hourly_steps = pd.read_csv(
    RAW_DATA_PATH / 'hourlySteps_merged.csv'
)

In [58]:
for df in [hourly_intensities, hourly_steps]:

    df.columns = (
        df.columns
        .str.strip()
        .str.replace(
            r'([a-z0-9])([A-Z])',
            r'\1_\2',
            regex=True
        )
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )

    df['id'] = df['id'].astype('string')

    df['activity_hour'] = pd.to_datetime(
        df['activity_hour'],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )

    df['date'] = df['activity_hour'].dt.normalize()

In [59]:
hourly_activity = (
    hourly_calories
    .merge(
        hourly_intensities,
        on=['id', 'activity_hour', 'date'],
        how='inner'
    )
    .merge(
        hourly_steps,
        on=['id', 'activity_hour', 'date'],
        how='inner'
    )
)

In [60]:
print("Integrated shape:", hourly_activity.shape)

print(
    "Unique participant-hour combinations:",
    hourly_activity[['id', 'activity_hour']].drop_duplicates().shape[0]
)

print(
    "Missing values:",
    hourly_activity.isna().sum().sum()
)

Integrated shape: (24084, 7)
Unique participant-hour combinations: 24084
Missing values: 0


In [61]:
hourly_activity.to_csv(
    PROCESSED_DATA_PATH / 'hourly_activity_clean.csv',
    index=False
)

In [62]:
hourly_activity

,id,activity_hour,calories,date,total_intensity,average_intensity,step_total
0,1503960366,2016-03-12 00:00:00,48,2016-03-12,0,0.000000,0
1,1503960366,2016-03-12 01:00:00,48,2016-03-12,0,0.000000,0
2,1503960366,2016-03-12 02:00:00,48,2016-03-12,0,0.000000,0
3,1503960366,2016-03-12 03:00:00,48,2016-03-12,0,0.000000,0
4,1503960366,2016-03-12 04:00:00,48,2016-03-12,0,0.000000,0
...,...,...,...,...,...,...,...
24079,8877689391,2016-04-12 04:00:00,73,2016-04-12,0,0.000000,0
24080,8877689391,2016-04-12 05:00:00,73,2016-04-12,0,0.000000,0
24081,8877689391,2016-04-12 06:00:00,96,2016-04-12,7,0.116667,209
24082,8877689391,2016-04-12 07:00:00,169,2016-04-12,26,0.433333,964


# CSV 5 — Minute-level activity integration

In [63]:
minute_calories = pd.read_csv(
    RAW_DATA_PATH / 'minuteCaloriesNarrow_merged.csv'
)

minute_intensities = pd.read_csv(
    RAW_DATA_PATH / 'minuteIntensitiesNarrow_merged.csv'
)

minute_mets = pd.read_csv(
    RAW_DATA_PATH / 'minuteMETsNarrow_merged.csv'
)

minute_steps = pd.read_csv(
    RAW_DATA_PATH / 'minuteStepsNarrow_merged.csv'
)

In [65]:
def prepare_minute_dataset(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.replace(
            r'([a-z0-9])([A-Z])',
            r'\1_\2',
            regex=True
        )
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )

    df['id'] = df['id'].astype('string')

    df['activity_minute'] = pd.to_datetime(
        df['activity_minute'],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )

    df['date'] = df['activity_minute'].dt.normalize()

    return df

In [66]:
minute_calories = prepare_minute_dataset(minute_calories)

minute_intensities = prepare_minute_dataset(minute_intensities)

minute_mets = prepare_minute_dataset(minute_mets)

minute_steps = prepare_minute_dataset(minute_steps)

In [67]:
key_cols = ['id', 'activity_minute']

datasets_to_check = {
    'calories': minute_calories,
    'intensities': minute_intensities,
    'mets': minute_mets,
    'steps': minute_steps
}

reference_keys = pd.MultiIndex.from_frame(
    minute_calories[key_cols]
)

for name, df in datasets_to_check.items():

    current_keys = pd.MultiIndex.from_frame(
        df[key_cols]
    )

    print(f"\n{name}")

    print("Same number of records:", len(df) == len(minute_calories))

    print("Same keys:", reference_keys.equals(current_keys))

    print("Missing from dataset:", len(reference_keys.difference(current_keys)))

    print("Additional keys:", len(current_keys.difference(reference_keys)))


calories
Same number of records: True
Same keys: True
Missing from dataset: 0
Additional keys: 0

intensities
Same number of records: True
Same keys: True
Missing from dataset: 0
Additional keys: 0

mets
Same number of records: True
Same keys: True
Missing from dataset: 0
Additional keys: 0

steps
Same number of records: True
Same keys: True
Missing from dataset: 0
Additional keys: 0


In [68]:
minute_activity = (
    minute_calories
    .merge(
        minute_intensities,
        on=['id', 'activity_minute', 'date'],
        how='inner'
    )
    .merge(
        minute_mets,
        on=['id', 'activity_minute', 'date'],
        how='inner'
    )
    .merge(
        minute_steps,
        on=['id', 'activity_minute', 'date'],
        how='inner'
    )
)

In [69]:
print("Shape:", minute_activity.shape)

print("Unique participants:", minute_activity['id'].nunique())

print(
    "Unique participant-minute keys:",
    minute_activity[key_cols].drop_duplicates().shape[0]
)

print("Missing values:", minute_activity.isna().sum().sum())

Shape: (1445040, 7)
Unique participants: 34
Unique participant-minute keys: 1445040
Missing values: 0


In [70]:
minute_activity.to_csv(
    PROCESSED_DATA_PATH / 'minute_activity_clean.csv',
    index=False
)

In [71]:
minute_activity

,id,activity_minute,calories,date,intensity,mets,steps
0,1503960366,2016-03-12 00:00:00,0.79730,2016-03-12,0,10,0
1,1503960366,2016-03-12 00:01:00,0.79730,2016-03-12,0,10,0
2,1503960366,2016-03-12 00:02:00,0.79730,2016-03-12,0,10,0
3,1503960366,2016-03-12 00:03:00,0.79730,2016-03-12,0,10,0
4,1503960366,2016-03-12 00:04:00,0.79730,2016-03-12,0,10,0
...,...,...,...,...,...,...,...
1445035,8877689391,2016-04-12 08:55:00,1.22480,2016-04-12,0,10,0
1445036,8877689391,2016-04-12 08:56:00,1.22480,2016-04-12,0,10,0
1445037,8877689391,2016-04-12 08:57:00,1.22480,2016-04-12,0,10,0
1445038,8877689391,2016-04-12 08:58:00,1.34728,2016-04-12,0,11,0


# CSV 6 — heartrate_seconds_merged.csv

In [72]:
heartrate = pd.read_csv(
    RAW_DATA_PATH / 'heartrate_seconds_merged.csv'
)

In [74]:
heartrate.columns = (
    heartrate.columns
    .str.strip()
    .str.replace(
        r'([a-z0-9])([A-Z])',
        r'\1_\2',
        regex=True
    )
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

In [75]:
heartrate['id'] = heartrate['id'].astype('string')

heartrate['time'] = pd.to_datetime(
    heartrate['time'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)

In [76]:
heartrate.rename(
    columns={'value': 'heart_rate'},
    inplace=True
)

In [77]:
heartrate['date'] = heartrate['time'].dt.normalize()

In [78]:
heartrate_clean = heartrate.copy()

In [79]:
heartrate_clean['hour'] = (
    heartrate_clean['time'].dt.hour
)

In [80]:
print("Shape:", heartrate_clean.shape)

print("Unique participants:", heartrate_clean['id'].nunique())

print("Invalid timestamps:", heartrate_clean['time'].isna().sum())

print("Missing values:", heartrate_clean.isna().sum().sum())

print("Heart-rate column:", heartrate_clean['heart_rate'].dtype)

Shape: (1154681, 5)
Unique participants: 14
Invalid timestamps: 0
Missing values: 0
Heart-rate column: int64


In [81]:
heartrate_clean.to_csv(
    PROCESSED_DATA_PATH / 'heartrate_seconds_clean.csv',
    index=False
)

In [82]:
from pathlib import Path
import pandas as pd

processed_files = {
    "daily_activity": "daily_activity_clean.csv",
    "sleep": "minute_sleep_clean.csv",
    "weight": "weight_clean.csv",
    "hourly_activity": "hourly_activity_clean.csv",
    "minute_activity": "minute_activity_clean.csv",
    "heartrate": "heartrate_seconds_clean.csv"
}

for name, filename in processed_files.items():

    file_path = PROCESSED_DATA_PATH / filename

    if file_path.exists():

        df = pd.read_csv(file_path)

        print(f"\n{'='*55}")
        print(name)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

    else:
        print(f"\nMISSING: {filename}")


daily_activity
Shape: (457, 19)
Columns: ['id', 'activity_date', 'total_steps', 'total_distance', 'tracker_distance', 'logged_activities_distance', 'very_active_distance', 'moderately_active_distance', 'light_active_distance', 'sedentary_active_distance', 'very_active_minutes', 'fairly_active_minutes', 'lightly_active_minutes', 'sedentary_minutes', 'calories', 'total_recorded_minutes', 'is_complete_day', 'component_distance_sum', 'distance_discrepancy']

sleep
Shape: (198034, 5)
Columns: ['id', 'date', 'value', 'log_id', 'calendar_date']

weight
Shape: (33, 7)
Columns: ['id', 'date', 'weight_kg', 'weight_pounds', 'bmi', 'is_manual_report', 'log_id']

hourly_activity
Shape: (24084, 7)
Columns: ['id', 'activity_hour', 'calories', 'date', 'total_intensity', 'average_intensity', 'step_total']

minute_activity
Shape: (1445040, 7)
Columns: ['id', 'activity_minute', 'calories', 'date', 'intensity', 'mets', 'steps']

heartrate
Shape: (1154681, 5)
Columns: ['id', 'time', 'heart_rate', 'date', 

In [83]:
validation = {
    "daily_activity": {
        "file": "daily_activity_clean.csv",
        "keys": ["id", "activity_date"]
    },
    "sleep": {
        "file": "minute_sleep_clean.csv",
        "keys": ["id", "log_id", "date"]
    },
    "weight": {
        "file": "weight_clean.csv",
        "keys": ["id", "log_id"]
    },
    "hourly_activity": {
        "file": "hourly_activity_clean.csv",
        "keys": ["id", "activity_hour"]
    },
    "minute_activity": {
        "file": "minute_activity_clean.csv",
        "keys": ["id", "activity_minute"]
    },
    "heartrate": {
        "file": "heartrate_seconds_clean.csv",
        "keys": ["id", "time"]
    }
}

for name, config in validation.items():

    df = pd.read_csv(
        PROCESSED_DATA_PATH / config["file"]
    )

    duplicate_keys = df.duplicated(
        subset=config["keys"]
    ).sum()

    print(
        f"{name}: "
        f"Rows={len(df):,}, "
        f"Duplicate keys={duplicate_keys:,}"
    )

daily_activity: Rows=457, Duplicate keys=0
sleep: Rows=198,034, Duplicate keys=0
weight: Rows=33, Duplicate keys=0
hourly_activity: Rows=24,084, Duplicate keys=0
minute_activity: Rows=1,445,040, Duplicate keys=0
heartrate: Rows=1,154,681, Duplicate keys=0
